In [1]:
pip install yfinance pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 4.0 MB/s  0:00:00m 3.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [yfinance]━━ 4/5 [yfinance]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import time

# ─────────────────────────────────────────────
# ETF LIST — 75 ETFs across major categories
# Each entry: (ticker, full_name, category, asset_class, fund_family)
# ─────────────────────────────────────────────
ETF_METADATA = [
    # US Broad Market Equity
    ("SPY",  "SPDR S&P 500 ETF Trust",                  "US Large Cap Blend",     "Equity",       "State Street"),
    ("IVV",  "iShares Core S&P 500 ETF",                "US Large Cap Blend",     "Equity",       "BlackRock"),
    ("VOO",  "Vanguard S&P 500 ETF",                    "US Large Cap Blend",     "Equity",       "Vanguard"),
    ("VTI",  "Vanguard Total Stock Market ETF",         "US Total Market",        "Equity",       "Vanguard"),
    ("ITOT", "iShares Core S&P Total US Stock Market",  "US Total Market",        "Equity",       "BlackRock"),
    ("SCHB", "Schwab US Broad Market ETF",              "US Total Market",        "Equity",       "Schwab"),
    ("IWB",  "iShares Russell 1000 ETF",                "US Large Cap Blend",     "Equity",       "BlackRock"),
    ("IWM",  "iShares Russell 2000 ETF",                "US Small Cap",           "Equity",       "BlackRock"),
    ("VB",   "Vanguard Small Cap ETF",                  "US Small Cap",           "Equity",       "Vanguard"),
    ("IJR",  "iShares Core S&P Small Cap ETF",          "US Small Cap",           "Equity",       "BlackRock"),

    # US Growth & Value
    ("QQQ",  "Invesco QQQ Trust",                       "US Large Cap Growth",    "Equity",       "Invesco"),
    ("IWF",  "iShares Russell 1000 Growth ETF",         "US Large Cap Growth",    "Equity",       "BlackRock"),
    ("VUG",  "Vanguard Growth ETF",                     "US Large Cap Growth",    "Equity",       "Vanguard"),
    ("SCHG", "Schwab US Large Cap Growth ETF",          "US Large Cap Growth",    "Equity",       "Schwab"),
    ("IWD",  "iShares Russell 1000 Value ETF",          "US Large Cap Value",     "Equity",       "BlackRock"),
    ("VTV",  "Vanguard Value ETF",                      "US Large Cap Value",     "Equity",       "Vanguard"),
    ("SCHV", "Schwab US Large Cap Value ETF",           "US Large Cap Value",     "Equity",       "Schwab"),
    ("DGRO", "iShares Core Dividend Growth ETF",        "US Dividend Growth",     "Equity",       "BlackRock"),
    ("VIG",  "Vanguard Dividend Appreciation ETF",      "US Dividend Growth",     "Equity",       "Vanguard"),
    ("DVY",  "iShares Select Dividend ETF",             "US High Dividend",       "Equity",       "BlackRock"),

    # US Sector ETFs
    ("XLK",  "Technology Select Sector SPDR",           "Technology",             "Equity",       "State Street"),
    ("XLF",  "Financial Select Sector SPDR",            "Financials",             "Equity",       "State Street"),
    ("XLV",  "Health Care Select Sector SPDR",          "Health Care",            "Equity",       "State Street"),
    ("XLE",  "Energy Select Sector SPDR",               "Energy",                 "Equity",       "State Street"),
    ("XLI",  "Industrial Select Sector SPDR",           "Industrials",            "Equity",       "State Street"),
    ("XLP",  "Consumer Staples Select Sector SPDR",     "Consumer Staples",       "Equity",       "State Street"),
    ("XLY",  "Consumer Discretionary Select Sector",    "Consumer Discretionary", "Equity",       "State Street"),
    ("XLU",  "Utilities Select Sector SPDR",            "Utilities",              "Equity",       "State Street"),
    ("XLRE", "Real Estate Select Sector SPDR",          "Real Estate",            "Equity",       "State Street"),
    ("XLB",  "Materials Select Sector SPDR",            "Materials",              "Equity",       "State Street"),

    # International Equity
    ("VEA",  "Vanguard FTSE Developed Markets ETF",     "International Developed","Equity",       "Vanguard"),
    ("IEFA", "iShares Core MSCI EAFE ETF",              "International Developed","Equity",       "BlackRock"),
    ("EFA",  "iShares MSCI EAFE ETF",                   "International Developed","Equity",       "BlackRock"),
    ("VWO",  "Vanguard FTSE Emerging Markets ETF",      "Emerging Markets",       "Equity",       "Vanguard"),
    ("EEM",  "iShares MSCI Emerging Markets ETF",       "Emerging Markets",       "Equity",       "BlackRock"),
    ("INDA", "iShares MSCI India ETF",                  "India Equity",           "Equity",       "BlackRock"),
    ("EWJ",  "iShares MSCI Japan ETF",                  "Japan Equity",           "Equity",       "BlackRock"),
    ("FXI",  "iShares China Large Cap ETF",             "China Equity",           "Equity",       "BlackRock"),
    ("EWZ",  "iShares MSCI Brazil ETF",                 "Brazil Equity",          "Equity",       "BlackRock"),
    ("ACWI", "iShares MSCI ACWI ETF",                   "Global All Cap",         "Equity",       "BlackRock"),

    # Fixed Income — Government
    ("SHY",  "iShares 1-3 Year Treasury Bond ETF",      "Short-Term Government",  "Fixed Income", "BlackRock"),
    ("IEF",  "iShares 7-10 Year Treasury Bond ETF",     "Intermediate Government","Fixed Income", "BlackRock"),
    ("TLT",  "iShares 20+ Year Treasury Bond ETF",      "Long-Term Government",   "Fixed Income", "BlackRock"),
    ("GOVT", "iShares US Treasury Bond ETF",            "US Treasury",            "Fixed Income", "BlackRock"),
    ("VGSH", "Vanguard Short-Term Treasury ETF",        "Short-Term Government",  "Fixed Income", "Vanguard"),
    ("VGIT", "Vanguard Intermediate-Term Treasury ETF", "Intermediate Government","Fixed Income", "Vanguard"),
    ("VGLT", "Vanguard Long-Term Treasury ETF",         "Long-Term Government",   "Fixed Income", "Vanguard"),
    ("TIPS", "iShares TIPS Bond ETF",                   "Inflation Protected",    "Fixed Income", "BlackRock"),
    ("VTIP", "Vanguard Short-Term Inflation Protected", "Inflation Protected",    "Fixed Income", "Vanguard"),

    # Fixed Income — Corporate & Aggregate
    ("AGG",  "iShares Core US Aggregate Bond ETF",      "US Aggregate Bond",      "Fixed Income", "BlackRock"),
    ("BND",  "Vanguard Total Bond Market ETF",          "US Aggregate Bond",      "Fixed Income", "Vanguard"),
    ("LQD",  "iShares iBoxx Investment Grade Corp ETF", "Investment Grade Corp",  "Fixed Income", "BlackRock"),
    ("VCIT", "Vanguard Intermediate-Term Corp Bond ETF","Investment Grade Corp",  "Fixed Income", "Vanguard"),
    ("HYG",  "iShares iBoxx High Yield Corp Bond ETF",  "High Yield",             "Fixed Income", "BlackRock"),
    ("JNK",  "SPDR Bloomberg High Yield Bond ETF",      "High Yield",             "Fixed Income", "State Street"),
    ("VCSH", "Vanguard Short-Term Corp Bond ETF",       "Short-Term Corp",        "Fixed Income", "Vanguard"),
    ("MUB",  "iShares National Muni Bond ETF",          "Municipal Bond",         "Fixed Income", "BlackRock"),
    ("EMB",  "iShares J.P. Morgan USD Emerging Markets","Emerging Market Bond",   "Fixed Income", "BlackRock"),

    # Multi-Asset & Balanced
    ("AOM",  "iShares Core Moderate Allocation ETF",    "Moderate Allocation",    "Multi-Asset",  "BlackRock"),
    ("AOR",  "iShares Core Growth Allocation ETF",      "Growth Allocation",      "Multi-Asset",  "BlackRock"),
    ("AOA",  "iShares Core Aggressive Allocation ETF",  "Aggressive Allocation",  "Multi-Asset",  "BlackRock"),
    ("AOK",  "iShares Core Conservative Allocation ETF","Conservative Allocation","Multi-Asset",  "BlackRock"),
    ("VSMGX","Vanguard LifeStrategy Moderate Growth",   "Moderate Allocation",    "Multi-Asset",  "Vanguard"),

    # Alternatives & Commodities
    ("GLD",  "SPDR Gold Shares",                        "Gold",                   "Commodity",    "State Street"),
    ("IAU",  "iShares Gold Trust",                      "Gold",                   "Commodity",    "BlackRock"),
    ("SLV",  "iShares Silver Trust",                    "Silver",                 "Commodity",    "BlackRock"),
    ("USO",  "United States Oil Fund",                  "Oil",                    "Commodity",    "USCF"),
    ("DJP",  "iPath Bloomberg Commodity Index",         "Broad Commodity",        "Commodity",    "Barclays"),
    ("PDBC", "Invesco Optimum Yield Diversified Cmdty", "Broad Commodity",        "Commodity",    "Invesco"),
    ("VNQ",  "Vanguard Real Estate ETF",                "REITs",                  "Real Estate",  "Vanguard"),
    ("IYR",  "iShares US Real Estate ETF",              "REITs",                  "Real Estate",  "BlackRock"),

    # ESG
    ("ESGU", "iShares MSCI USA ESG Optimized ETF",      "ESG US Equity",          "Equity",       "BlackRock"),
    ("ESGV", "Vanguard ESG US Stock ETF",               "ESG US Equity",          "Equity",       "Vanguard"),
    ("ESGE", "iShares MSCI EM ESG Optimized ETF",       "ESG Emerging Markets",   "Equity",       "BlackRock"),

    # Retirement / Target Date proxies
    ("SCHD", "Schwab US Dividend Equity ETF",           "US Dividend",            "Equity",       "Schwab"),
    ("NOBL", "ProShares S&P 500 Dividend Aristocrats",  "Dividend Aristocrats",   "Equity",       "ProShares"),
]

# ─────────────────────────────────────────────
# PULL PRICE HISTORY — 5 years
# ─────────────────────────────────────────────
END_DATE   = datetime.today()
START_DATE = END_DATE - timedelta(days=5*365)

tickers = [t[0] for t in ETF_METADATA]

print(f"Pulling price history for {len(tickers)} ETFs from {START_DATE.date()} to {END_DATE.date()}...")
print("This may take 1-2 minutes...\n")

raw = yf.download(
    tickers,
    start=START_DATE.strftime("%Y-%m-%d"),
    end=END_DATE.strftime("%Y-%m-%d"),
    auto_adjust=True,
    progress=True,
    threads=True
)

# Keep only Close prices, reshape to long format
close = raw["Close"].copy()
close.index.name = "Date"
close = close.reset_index()

price_long = close.melt(id_vars="Date", var_name="Ticker", value_name="Close_Price")
price_long.dropna(subset=["Close_Price"], inplace=True)
price_long["Date"] = pd.to_datetime(price_long["Date"])
price_long["Year"]  = price_long["Date"].dt.year
price_long["Month"] = price_long["Date"].dt.month
price_long["Month_Name"] = price_long["Date"].dt.strftime("%B")
price_long["Quarter"] = price_long["Date"].dt.quarter.map({1:"Q1",2:"Q2",3:"Q3",4:"Q4"})

print(f"\nPrice rows pulled: {len(price_long):,}")

# ─────────────────────────────────────────────
# CALCULATE RETURNS per ETF
# ─────────────────────────────────────────────
print("Calculating returns...")

returns_rows = []
today_prices = {}

for ticker in tickers:
    df_t = price_long[price_long["Ticker"] == ticker].sort_values("Date")
    if df_t.empty or len(df_t) < 20:
        continue

    latest_price = df_t["Close_Price"].iloc[-1]
    latest_date  = df_t["Date"].iloc[-1]
    today_prices[ticker] = latest_price

    def ret(days):
        cutoff = latest_date - timedelta(days=days)
        past = df_t[df_t["Date"] <= cutoff]
        if past.empty:
            return None
        past_price = past["Close_Price"].iloc[-1]
        return round((latest_price - past_price) / past_price * 100, 2)

    returns_rows.append({
        "Ticker":         ticker,
        "Latest_Price":   round(latest_price, 2),
        "Latest_Date":    latest_date.date(),
        "Return_1M_pct":  ret(30),
        "Return_3M_pct":  ret(90),
        "Return_6M_pct":  ret(180),
        "Return_1Y_pct":  ret(365),
        "Return_3Y_pct":  ret(3*365),
        "Return_5Y_pct":  ret(5*365),
    })

returns_df = pd.DataFrame(returns_rows)

# ─────────────────────────────────────────────
# METADATA TABLE
# ─────────────────────────────────────────────
meta_df = pd.DataFrame(ETF_METADATA, columns=[
    "Ticker","ETF_Name","Category","Asset_Class","Fund_Family"
])

# ─────────────────────────────────────────────
# MERGE METADATA + RETURNS → summary table
# ─────────────────────────────────────────────
summary_df = meta_df.merge(returns_df, on="Ticker", how="left")

# ─────────────────────────────────────────────
# MERGE METADATA → price history (main fact table)
# ─────────────────────────────────────────────
fact_df = price_long.merge(meta_df, on="Ticker", how="left")

# Daily return within each ETF
fact_df = fact_df.sort_values(["Ticker","Date"])
fact_df["Daily_Return_pct"] = (
    fact_df.groupby("Ticker")["Close_Price"]
    .pct_change() * 100
).round(4)

# Rolling 30-day volatility (std of daily returns)
fact_df["Volatility_30D"] = (
    fact_df.groupby("Ticker")["Daily_Return_pct"]
    .transform(lambda x: x.rolling(30).std())
).round(4)

# ─────────────────────────────────────────────
# EXPORT
# ─────────────────────────────────────────────
out_fact    = "/Users/piyushsaraf/Desktop/etf_price_history.csv"
out_summary = "/Users/piyushsaraf/Desktop/etf_summary_returns.csv"

fact_df.to_csv(out_fact, index=False)
summary_df.to_csv(out_summary, index=False)

print(f"\n✅ Done!")
print(f"   Price history  → {out_fact}  ({len(fact_df):,} rows)")
print(f"   Summary/returns→ {out_summary}  ({len(summary_df)} ETFs)")
print(f"\nColumns in price history:")
print(list(fact_df.columns))
print(f"\nColumns in summary:")
print(list(summary_df.columns))

Pulling price history for 76 ETFs from 2021-05-30 to 2026-05-29...
This may take 1-2 minutes...



[*********************100%***********************]  76 of 76 completed



Price rows pulled: 95,301
Calculating returns...

✅ Done!
   Price history  → /Users/piyushsaraf/Desktop/etf_price_history.csv  (95,301 rows)
   Summary/returns→ /Users/piyushsaraf/Desktop/etf_summary_returns.csv  (76 ETFs)

Columns in price history:
['Date', 'Ticker', 'Close_Price', 'Year', 'Month', 'Month_Name', 'Quarter', 'ETF_Name', 'Category', 'Asset_Class', 'Fund_Family', 'Daily_Return_pct', 'Volatility_30D']

Columns in summary:
['Ticker', 'ETF_Name', 'Category', 'Asset_Class', 'Fund_Family', 'Latest_Price', 'Latest_Date', 'Return_1M_pct', 'Return_3M_pct', 'Return_6M_pct', 'Return_1Y_pct', 'Return_3Y_pct', 'Return_5Y_pct']
